# DoubleTM Convex Hull

Build convex-hull plots from `doubleTM_dft_2nd_relax_summary.csv` for each `tm_pair`.
Each `elec_*` is treated as a distinct ordering.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.lines import Line2D
from scipy.spatial import ConvexHull
from pymatgen.core.composition import Composition

plt.rcParams.update({
    "font.size": 15,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "legend.title_fontsize": 15,
})

EXCLUDE_ORDERINGS = {}

na_map = {
    "AJT_a": 1/7, "AJT_b": 1/4, "AJT_c": 1/4, "AJT_d": 1/3, "AJT_e": 1/3, 
    "AJT_f": 2/5, "AJT_g": 1/2, "AJT_h": 3/5, "AJT_i": 5/8, "AJT_j": 5/8,
    "AJT_k": 2/3, "AJT_l": 2/3, "AJT_m": 2/3, "AJT_n": 3/4, "AJT_o": 3/4,
    "AJT_p": 5/6, "AJT_q": 7/8, "AJT_r": 8/9,
    "JLK_b": 3/8, "JLK_c": 7/10, "JLK_theta_4": 4/5, "JLK_theta_5": 5/6,
    "JLK_delta": 2/3, "JLK_eta": 4/7, "JLK_zeta": 1/2, "JLK_zeta2": 2/5,
    "JLK_zeta3": 3/7, "JLK_zeta6": 6/11,
    "CD_a": 1/2, "JMAS_a": 1/3, "JMAS_b": 2/5, "JMAS_e": 1, "XC_a": 1/3,
    "XC_b": 1/2, "JMAS_c": 1/2, "JMAS_d": 2/3, "XL_a": 5/8
}

def load_csvs():
    sources = [
        (Path("../data/doubleTM_dft_tighter.csv"), "dft_summary"),
        # (Path("../data/doubleTM_dft_summary.csv"), "dft_summary"),
        # (Path("../data/doubleTM_dft_2nd_relax_summary.csv"), "dft_2nd_relax_summary"),
    ]
    frames = []
    for p, tag in sources:
        if p.exists():
            d = pd.read_csv(p)
            d["source_tag"] = tag
            frames.append(d)
            print(f"Loaded: {p} ({len(d)} rows)")
    if not frames:
        raise FileNotFoundError("No doubleTM summary CSV found.")
    return pd.concat(frames, ignore_index=True)

def short_name(ordering: str) -> str:
    s = str(ordering).replace("_w_vac", "")
    if s.startswith(("O1_", "O3_", "P3_")):
        rest = s.split("_", 1)[1]
        if rest in {"full", "empty"}:
            return rest
    return s

def format_annotation_label(ordering_elec: str) -> str:
    s = str(ordering_elec).replace("__elec_", "_")
    for suffix in ("__dft_summary", "__dft_2nd_relax_summary", "__unknown_src"):
        if s.endswith(suffix):
            s = s[:-len(suffix)]
            break
    return s

def energy_per_TMO2(row):
    formula = row["formula"]
    natoms = float(row["natoms"])
    tm_pair = str(row["tm_pair"])
    tm_list = [x for x in tm_pair.split("_") if x]

    if not tm_list or pd.isna(formula) or pd.isna(natoms) or natoms <= 0:
        return np.nan

    comp_norm = Composition(formula)
    n_norm = comp_norm.num_atoms
    if n_norm <= 0:
        return np.nan

    scale = natoms / n_norm
    n_tmo2_units = sum(comp_norm[tm] for tm in tm_list) * scale
    if n_tmo2_units <= 0:
        return np.nan

    if "energy_eV" in row and not pd.isna(row["energy_eV"]):
        e_tot = float(row["energy_eV"])
    elif "energy_per_atom_eV" in row and not pd.isna(row["energy_per_atom_eV"]):
        e_tot = float(row["energy_per_atom_eV"]) * natoms
    else:
        return np.nan

    return e_tot / n_tmo2_units

def lower_hull_indices_scipy(xs, ys):
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    if len(xs) == 0:
        return []

    unique_indices = []
    for x0 in sorted(set(xs)):
        idxs = np.where(np.isclose(xs, x0))[0]
        unique_indices.append(idxs[np.argmin(ys[idxs])])

    xs_u = xs[unique_indices]
    ys_u = ys[unique_indices]
    if len(xs_u) <= 2:
        return sorted(unique_indices, key=lambda i: xs[i])

    pts = np.column_stack([xs_u, ys_u])
    hull = ConvexHull(pts)
    verts_sorted = sorted(hull.vertices, key=lambda i: (xs_u[i], ys_u[i]))

    lower_u = []
    for i in verts_sorted:
        while len(lower_u) >= 2:
            j = lower_u[-2]
            k = lower_u[-1]
            x1, y1 = xs_u[j], ys_u[j]
            x2, y2 = xs_u[k], ys_u[k]
            x3, y3 = xs_u[i], ys_u[i]
            cross = (x2 - x1) * (y3 - y1) - (y2 - y1) * (x3 - x1)
            if cross <= 0:
                lower_u.pop()
            else:
                break
        lower_u.append(i)

    lower_idx = [unique_indices[i] for i in lower_u]
    return sorted(lower_idx, key=lambda i: xs[i])

df = load_csvs()

mask = pd.Series(True, index=df.index)
if "status" in df.columns:
    mask &= df["status"] == "OK"
if "n_custodian_errors" in df.columns:
    mask &= pd.to_numeric(df["n_custodian_errors"], errors="coerce").fillna(0) == 0
if "converged" in df.columns:
    c = df["converged"]
    if c.dtype == object:
        c = c.astype(str).str.lower() == "true"
    mask &= c
df = df[mask].copy()

df["natoms"] = pd.to_numeric(df["natoms"], errors="coerce")
df["energy_eV"] = pd.to_numeric(df.get("energy_eV"), errors="coerce")
if "energy_per_atom_eV" in df.columns:
    df["energy_per_atom_eV"] = pd.to_numeric(df["energy_per_atom_eV"], errors="coerce")

df["ordering_short"] = df["ordering"].apply(short_name)
if EXCLUDE_ORDERINGS:
    df = df[~df["ordering_short"].isin(EXCLUDE_ORDERINGS)].copy()

df["xNa"] = df["ordering_short"].map(na_map)
mask_full = df["ordering_short"] == "full"
mask_empty = df["ordering_short"] == "empty"
df.loc[mask_full, "xNa"] = 1.0
df.loc[mask_empty, "xNa"] = 0.0
df = df.dropna(subset=["xNa"]).copy()

df["E_TMO2_eV"] = df.apply(energy_per_TMO2, axis=1)
df = df.dropna(subset=["E_TMO2_eV"]).copy()

elec_col = df["elec"].fillna("elec_unknown").astype(str)
src_col = df["source_tag"].fillna("unknown_src").astype(str)
df["ordering_elec"] = df["ordering_short"].astype(str) + "__" + elec_col + "__" + src_col

tm_pairs = sorted(df["tm_pair"].dropna().unique())
out_paths = []
for tm_pair in tm_pairs:
    df_tm = df[df["tm_pair"] == tm_pair].copy()
    if df_tm.empty:
        continue

    df_pts = (
        df_tm
        .groupby(["ordering_elec", "phase", "xNa"], as_index=False)
        .agg({"E_TMO2_eV": "mean"})
    )
    if len(df_pts) < 2:
        continue

    xs = df_pts["xNa"].to_numpy(dtype=float)
    ys = df_pts["E_TMO2_eV"].to_numpy(dtype=float)
    hull_idx = lower_hull_indices_scipy(xs, ys)
    if not hull_idx:
        continue

    hull_mask = np.zeros(len(df_pts), dtype=bool)
    hull_mask[hull_idx] = True

    if len(hull_idx) >= 2:
        iL, iR = hull_idx[0], hull_idx[-1]
        xL, yL = xs[iL], ys[iL]
        xR, yR = xs[iR], ys[iR]
        if xR != xL:
            y_ref = yL + (yR - yL) * (xs - xL) / (xR - xL)
            ys_rel = ys - y_ref
        else:
            ys_rel = ys - ys.min()
    else:
        ys_rel = ys - ys.min()
    df_pts["E_rel_TMO2_eV"] = ys_rel

    fig, ax = plt.subplots(figsize=(6, 4.5))
    is_o3 = df_pts["phase"].astype(str).str.contains("O3")
    is_p3 = df_pts["phase"].astype(str).str.contains("P3")
    is_o1 = df_pts["phase"].astype(str).str.contains("O1")

    for phase_mask, marker in [(is_o3, "o"), (is_p3, "s"), (is_o1, "^")]:
        m_non = phase_mask & (~hull_mask)
        m_hul = phase_mask & hull_mask
        if m_non.any():
            ax.scatter(df_pts.loc[m_non, "xNa"], df_pts.loc[m_non, "E_rel_TMO2_eV"],
                       marker=marker, s=130, facecolors="none", edgecolors="red", linewidths=1.7)
        if m_hul.any():
            ax.scatter(df_pts.loc[m_hul, "xNa"], df_pts.loc[m_hul, "E_rel_TMO2_eV"],
                       marker=marker, s=170, facecolors="none", edgecolors="black", linewidths=2.2)

    if len(hull_idx) > 1:
        xs_arr = df_pts["xNa"].to_numpy()
        e_arr = df_pts["E_rel_TMO2_eV"].to_numpy()
        hx = [xs_arr[i] for i in hull_idx]
        hy = [e_arr[i] for i in hull_idx]
        ax.plot(hx, hy, color="black", linewidth=2)

        y_span = max(hy) - min(hy) if len(hy) > 1 else 0.01
        y_offset = 0.03 * y_span
        for i in hull_idx:
            row_pt = df_pts.iloc[i]
            label = format_annotation_label(row_pt["ordering_elec"])
            ax.text(
                xs_arr[i] - 0.05,
                e_arr[i] - y_offset * 4.4,
                label,
                fontsize=11,
                ha="left",
                va="bottom",
                rotation=0,
            )

    handles = [
        Line2D([], [], marker="o", linestyle="None", markerfacecolor="none", markeredgecolor="black", markersize=10, label="O3"),
        Line2D([], [], marker="s", linestyle="None", markerfacecolor="none", markeredgecolor="black", markersize=10, label="P3"),
        Line2D([], [], marker="^", linestyle="None", markerfacecolor="none", markeredgecolor="black", markersize=10, label="O1"),
        Line2D([], [], marker="o", linestyle="None", markerfacecolor="none", markeredgecolor="black", markersize=10, label="hull"),
        Line2D([], [], marker="o", linestyle="None", markerfacecolor="none", markeredgecolor="red", markersize=10, label="non-hull"),
    ]
    ax.legend(handles=handles, frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0.0)
    tm1, tm2 = tm_pair.split("_")
    ax.set_xlabel(f"x in Na$_x${tm1}$_{{0.5}}${tm2}$_{{0.5}}$O$_2$")
    ax.set_ylabel("Formation Energy (eV/TMO$_2$)")
    ax.set_xlim(-0.02, 1.02)
    y_vals = df_pts["E_rel_TMO2_eV"].to_numpy(dtype=float)
    y_min = float(np.min(y_vals))
    y_margin = 0.05 * (1.0 - y_min if 1.0 > y_min else 1.0)
    ax.set_ylim(y_min - y_margin, 0.1)
    ax.grid(True, linestyle="--", alpha=0.2)
    ax.set_axisbelow(True)

    outpath = Path(f"../data/figures/doubleTM/convexhull_tighterConv_{tm_pair}.pdf")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    out_paths.append(str(outpath))

print(f"Saved {len(out_paths)} convex-hull plots")
out_paths[:10]

Loaded: ../data/doubleTM_dft_tighter.csv (820 rows)
Saved 10 convex-hull plots


['../data/figures/doubleTM/convexhull_tighterConv_Co_Ni.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Fe_Co.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Fe_Ni.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Mn_Co.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Mn_Fe.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Mn_Ni.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Ti_Co.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Ti_Fe.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Ti_Mn.pdf',
 '../data/figures/doubleTM/convexhull_tighterConv_Ti_Ni.pdf']